In [ ]:
import logging
# from src.config import load_config
# from src.data.loader import load_datasets
from src.models.codegen_loader_small_model import CodeGenModelWrapper
from src.models.finetuned_LoRa import train_codegen_lora
# from src.tasks.program_gen import run_program_gen
# from src.tasks.doc_gen import run_doc_gen
# from src.tasks.text_to_sql import run_text_to_sql
# from src.tasks.commit_gen import run_commit_gen
# from src.indexing.embeddings import generate_code_embeddings
# from src.indexing.semantic_index import FAISSIndexManager
# from src.indexing.ast_index import ASTIndexManager
from src.rag.rag_retriever import TopKRetriever
from src.rag.rag_pipeline import RAGPipeline
from src.evaluation.comparator import compare_architectures
from src.evaluation.visualization import save_plots

In [ ]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s - [%(levelname)s] - %(message)s")

In [ ]:
def main():
    print("=" * 70)
    print("STARTING END-TO-END CODEGEN-350M-MULTI PIPELINE")
    print("=" * 70)

    config = load_config()

    datasets = UnifiedDatasetLoader.load_datasets(config)

    model = CodeGenMultiModelWrapper(
        model_name=config["model"]["name_or_path"],
        device=config["project"]["device"],
        use_lora=True,
        lora_kwargs=config.get("lora", {})
    )

    training_texts = ["def binary_search(arr, t): return -1"]
    train_codegen_lora(model, training_texts, config["outputs"]["checkpoints_dir"], epochs=1)

    p_out = run_program_generation(model)
    d_out = run_doc_generation(model)
    s_out = run_text_to_sql_generation(model)
    c_out = run_commit_generation(model)

    code_corpus = [
        "def quicksort(arr): return arr",
        "SELECT * FROM users WHERE active = 1;",
        "def add(a, b): return a + b"
    ]
    embeddings = generate_code_lm_embeddings(model, code_corpus)

    faiss_mgr = FAISSIndexManager()
    faiss_mgr.build_index(embeddings, code_corpus)

    ast_mgr = TreeSitterASTIndexer(model)
    ast_mgr.build_ast_index(code_corpus)

    retriever = TopKRetriever(model, faiss_mgr)
    rag_pipe = RAGPipeline(model)

    query = "Write a Python function for sorting"
    ctx = retriever.retrieve(query, top_k=config["indexing"]["top_k"])
    rag_result = rag_pipe.execute(query, ctx)

    print("\n--- FINAL RAG GENERATED OUTPUT ---")
    print(rag_result)
    print("----------------------------------\n")

    comparison = ArchitectureComparator.compare_architectures()
    ResultsVisualizer.save_metrics(comparison, f"{config['outputs']['metrics_dir']}/results.json")
    ResultsVisualizer.generate_chart(comparison, config['outputs']['plots_dir'])

    print("=" * 70)
    print("PIPELINE EXECUTION COMPLETE")
    print("=" * 70)

In [ ]:
if __name__ == "__main__":
    main()